In [1]:
from pydantic import BaseModel
from typing import List



# BUILDING PYDANTIC SCHEMA FOR THE SEASHELL STORE

class PurchasedProduct(BaseModel):
    product_name: str
    quantity: int
    price: float


class CustomerOrder(BaseModel):
    order_id: int
    status: str
    total: float
    products: List[PurchasedProduct]


class CustomerDocument(BaseModel):
    customer_id: int
    name: str
    email: str
    city: str
    orders: List[CustomerOrder]

In [4]:
import sqlite3

conn = sqlite3.connect("../../shop.db")
cursor = conn.cursor()


# BUILDING LIST OF COMPLETE CUSTOMER DOCUMENTS FROM DB DATA

cursor.execute("""
    SELECT id, name, email, city
    FROM customers
""")

customers = cursor.fetchall()


documents = []


for customer_id, name, email, city in customers:

    cursor.execute("""
        SELECT
            o.id,
            o.status,
            o.total,
            oi.quantity,
            oi.price,
            p.name
        FROM orders o
        JOIN order_items oi
            ON o.id = oi.order_id
        JOIN products p
            ON oi.product_id = p.id
        WHERE o.customer_id = ?
        ORDER BY o.id
    """, (customer_id,))

    rows = cursor.fetchall()


    orders = {}

    for (
        order_id,
        status,
        total,
        quantity,
        price,
        product_name,
    ) in rows:

        if order_id not in orders:
            orders[order_id] = CustomerOrder(
                order_id=order_id,
                status=status,
                total=total,
                products=[],
            )

        orders[order_id].products.append(
            PurchasedProduct(
                product_name=product_name,
                quantity=quantity,
                price=price,
            )
        )


    customer_document = CustomerDocument(
        customer_id=customer_id,
        name=name,
        email=email,
        city=city,
        orders=list(orders.values()),
    )

    documents.append(customer_document)


conn.close()

In [5]:
# Convert them to JSON

import json

data = [
    document.model_dump()
    for document in documents
]

with open("customers.json", "w") as f:
    json.dump(data, f, indent=2)

### NOW that we have a json document for our data we can start data ingestion and creating documents

In [7]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="customers.json",
    jq_schema=".[]",
    text_content=False,
)

docs = loader.load()

In [8]:
for doc in docs[:2]:
    print(doc)
    print("=" * 80)

page_content='{"customer_id": 1, "name": "Sadaf Iqbal", "email": "sadafiqbal4911@gmail.com", "city": "Kabirwala", "orders": [{"order_id": 1, "status": "completed", "total": 661, "products": [{"product_name": "Mechanical Keyboard", "quantity": 3, "price": 120}, {"product_name": "Bluetooth Speaker", "quantity": 1, "price": 85}, {"product_name": "Ethernet Cable", "quantity": 3, "price": 12}, {"product_name": "Webcam", "quantity": 2, "price": 90}]}, {"order_id": 2, "status": "processing", "total": 615, "products": [{"product_name": "Laptop Stand", "quantity": 1, "price": 75}, {"product_name": "1080p Monitor", "quantity": 3, "price": 180}]}, {"order_id": 3, "status": "processing", "total": 40, "products": [{"product_name": "Smartphone Stand", "quantity": 2, "price": 20}]}, {"order_id": 4, "status": "shipped", "total": 120, "products": [{"product_name": "USB-C Hub", "quantity": 2, "price": 60}]}]}' metadata={'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/cust

In [ ]:
# This is what current metadata looks like

metadata={'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'seq_num': 1}

### OK we have the documents but the metadata isn't that useful so we fine tune it

In [9]:
def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["customer_id"] = record["customer_id"]
    metadata["customer_name"] = record["name"]
    metadata["city"] = record["city"]
    metadata["email"] = record["email"]

    metadata["order_count"] = len(record["orders"])

    return metadata

In [12]:
loader = JSONLoader(
    file_path="customers.json",
    jq_schema=".[]",
    content_key=None,
    metadata_func=metadata_func,
    text_content=False,
)

In [13]:
docs = loader.load()

In [14]:
for doc in docs[:2]:
    print("PAGE CONTENT:")
    print(doc.page_content)

    print("\nMETADATA:")
    print(doc.metadata)

    print("=" * 80)

PAGE CONTENT:
{"customer_id": 1, "name": "Sadaf Iqbal", "email": "sadafiqbal4911@gmail.com", "city": "Kabirwala", "orders": [{"order_id": 1, "status": "completed", "total": 661, "products": [{"product_name": "Mechanical Keyboard", "quantity": 3, "price": 120}, {"product_name": "Bluetooth Speaker", "quantity": 1, "price": 85}, {"product_name": "Ethernet Cable", "quantity": 3, "price": 12}, {"product_name": "Webcam", "quantity": 2, "price": 90}]}, {"order_id": 2, "status": "processing", "total": 615, "products": [{"product_name": "Laptop Stand", "quantity": 1, "price": 75}, {"product_name": "1080p Monitor", "quantity": 3, "price": 180}]}, {"order_id": 3, "status": "processing", "total": 40, "products": [{"product_name": "Smartphone Stand", "quantity": 2, "price": 20}]}, {"order_id": 4, "status": "shipped", "total": 120, "products": [{"product_name": "USB-C Hub", "quantity": 2, "price": 60}]}]}

METADATA:
{'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/cus

In [ ]:
# This is how after fine tuning it looks like


METADATA= {'source': '/home/workspace/learning/RAG/notebooks/seashell_store_data_ingestion/customers.json', 'seq_num': 1, 'customer_id': 1, 'customer_name': 'Sadaf Iqbal', 'city': 'Kabirwala', 'email': 'sadafiqbal4911@gmail.com', 'order_count': 4}